In [ ]:
# 单元格1：导入必要的库和设置基础路径
import h5py
import numpy as np
import os
import pickle
import scipy.io
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import datetime
import glob
import shutil
from sklearn.model_selection import train_test_split

# 设置基础路径
base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data'
data_path = os.path.join(base_dir, 'DATA/TRAIN38.mat')
output_base_dir = os.path.join(base_dir, 'processed_data')

# 创建输出目录
os.makedirs(output_base_dir, exist_ok=True)

print(f"数据路径: {data_path}")
print(f"输出目录: {output_base_dir}")

In [ ]:
# 单元格2（修正版）：加载数据并确保正确处理age字段
# 加载TRAIN38.mat文件
f = h5py.File(data_path, 'r')
arrays = {}
for k, v in f.items():
    print(f"加载键: {k}, 形状: {v.shape}")
    arrays[k] = np.array(v)
f.close()

# 提取数据并转置 - 特别注意age字段的处理
data = arrays['data'].transpose()  # 体素特征数据
region = arrays['region'].transpose()  # 102维的区域标签
prob_idx = arrays['prob_idx'].transpose()  # 病人ID

# 处理age字段 - 确保正确加载和转置
if 'age' in arrays:
    age_raw = arrays['age']
    # 根据age的维度决定是否需要转置
    if len(age_raw.shape) > 1 and age_raw.shape[0] > 1:
        age = age_raw.transpose()
        print("age数据已转置")
    else:
        age = age_raw
        print("age数据未转置，保持原始形状")
    
    print(f"age数据形状: {age.shape}")
    print(f"数据中的一些age值: {age[:10]}")
else:
    age = None
    print("未找到age数据")

# 数据基本信息
print(f"数据形状: {data.shape}")
print(f"标签形状: {region.shape}")
print(f"病人索引形状: {prob_idx.shape}")
if age is not None:
    print(f"年龄数据形状: {age.shape}")
else:
    print("未找到年龄数据")

# 分析唯一的病人ID
unique_prob_idx = np.unique(prob_idx)
print(f"唯一病人ID: {unique_prob_idx}")
print(f"病人总数: {len(unique_prob_idx)}个")

# 分析区域标签分布
if region.ndim == 2:
    # 计算每个区域标签中有多少个体素
    label_counts = np.sum(region, axis=0)
    active_regions = []
    for i in range(region.shape[1]):
        if label_counts[i] > 0:
            active_regions.append(i)
            print(f"区域 {i}: {label_counts[i]} 个体素")
    
    print(f"活跃区域数量: {len(active_regions)}")
    
    # 计算每个体素被分配到了几个区域
    region_per_voxel = np.sum(region, axis=1)
    unique_counts, count_freqs = np.unique(region_per_voxel, return_counts=True)
    for count, freq in zip(unique_counts, count_freqs):
        print(f"{count} 个区域标签的体素数量: {freq}")
else:
    print("标签不是二维的，无法分析区域分布")

# 分析每个病人的样本数量
for idx in unique_prob_idx:
    count = np.sum(prob_idx == idx)
    print(f"病人 {idx}: {count}个样本")

In [ ]:
# 单元格3：按照病人ID划分数据集
# 设置随机种子确保结果可重现
np.random.seed(42)

# 病人划分
test_patients = [38]  # 第38号病人直接进入测试集
remaining_patients = [i for i in range(1, 38)]  # 剩余37个病人

# 随机选择7个病人加入测试集
additional_test_patients = np.random.choice(remaining_patients, 7, replace=False)
test_patients.extend(additional_test_patients)

# 剩余的30个病人用于训练和验证
train_val_patients = [p for p in remaining_patients if p not in additional_test_patients]

print(f"测试集病人 ({len(test_patients)}个): {sorted(test_patients)}")
print(f"训练和验证集病人 ({len(train_val_patients)}个): {sorted(train_val_patients)}")

# 根据病人ID划分数据
test_indices = np.where(np.isin(prob_idx, test_patients))[0]
train_val_indices = np.where(np.isin(prob_idx, train_val_patients))[0]

# 提取测试集 - 完整保留所有原始数据结构
test_data = data[test_indices]
test_regions = region[test_indices]
test_prob_idx = prob_idx[test_indices]
test_age = age[test_indices] if age is not None else None

# 提取训练和验证集 - 完整保留所有原始数据结构
train_val_data = data[train_val_indices]
train_val_regions = region[train_val_indices]
train_val_prob_idx = prob_idx[train_val_indices]
train_val_age = age[train_val_indices] if age is not None else None

print(f"测试集样本数: {len(test_data)}")
print(f"训练和验证集样本数: {len(train_val_data)}")

# 验证一一对应关系
print("\n验证数据分割后的一一对应关系:")
print(f"测试集: 数据形状 {test_data.shape}, 标签形状 {test_regions.shape}, 病人ID形状 {test_prob_idx.shape}")
if test_age is not None:
    print(f"测试集年龄数据形状: {test_age.shape}")
print(f"训练和验证集: 数据形状 {train_val_data.shape}, 标签形状 {train_val_regions.shape}, 病人ID形状 {train_val_prob_idx.shape}")
if train_val_age is not None:
    print(f"训练和验证集年龄数据形状: {train_val_age.shape}")

# 保存病人分配信息
patient_allocation = {
    "test_patients": sorted(test_patients),
    "train_val_patients": sorted(train_val_patients)
}

In [ ]:
# 单元格4：创建和保存StandardScaler
# 功能：使用所有训练和验证数据拟合StandardScaler并保存

# 使用训练和验证集数据拟合StandardScaler
print(f"使用 {train_val_data.shape[0]} 个样本拟合Scaler...")
scaler = StandardScaler()
scaler.fit(train_val_data)

# 保存Scaler
scaler_path = os.path.join(output_base_dir, 'data_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"Scaler已拟合并保存到: {scaler_path}")
print(f"Scaler均值形状: {scaler.mean_.shape}")
print(f"Scaler方差形状: {scaler.var_.shape}")

# 创建输出目录
train_dir = os.path.join(output_base_dir, 'train')
val_dir = os.path.join(output_base_dir, 'val')
test_dir = os.path.join(output_base_dir, 'test')
npy_dir = os.path.join(output_base_dir, 'npy')  # 用于存储npy格式
mat_dir = os.path.join(output_base_dir, 'mat')  # 用于存储mat格式

for directory in [train_dir, val_dir, test_dir, npy_dir, mat_dir]:
    if not os.path.exists(directory):
        os.makedirs(directory)
    for subdir in ['npy', 'mat']:
        subdir_path = os.path.join(directory, subdir)
        if not os.path.exists(subdir_path):
            os.makedirs(subdir_path)

In [ ]:
# 单元格5（修正版）：按区域标签分组处理数据，确保正确保存age数据
# 功能：将训练和验证数据按区域标签分组，按6:2比例拆分，同时保存npy和mat格式

# 找出哪些区域标签有数据
active_regions = []
for i in range(train_val_regions.shape[1]):
    if np.sum(train_val_regions[:, i]) > 0:
        active_regions.append(i)

print(f"活跃的区域标签数: {len(active_regions)}")
print(f"活跃的区域标签: {active_regions}")

# 将训练和验证数据按6:2比例分割
train_ratio = 0.6
np.random.seed(42)  # 确保结果可重现

# 对每个活跃区域进行处理
for region_idx in tqdm(active_regions, desc="处理区域标签"):
    # 找出该区域的所有体素
    indices = np.where(train_val_regions[:, region_idx] == 1)[0]
    
    if len(indices) == 0:
        print(f"区域 {region_idx} 没有数据，跳过")
        continue
    
    # 提取该区域的所有数据
    region_data = train_val_data[indices]
    region_labels = train_val_regions[indices]
    region_prob_idx = train_val_prob_idx[indices]
    region_age = train_val_age[indices] if train_val_age is not None else None
    
    # 随机打乱索引
    shuffle_indices = np.random.permutation(len(indices))
    
    # 按照打乱的索引重排数据
    region_data = region_data[shuffle_indices]
    region_labels = region_labels[shuffle_indices]
    region_prob_idx = region_prob_idx[shuffle_indices]
    if region_age is not None:
        region_age = region_age[shuffle_indices]
    
    # 按6:2比例拆分
    train_size = int(len(region_data) * train_ratio)
    
    # 训练集
    train_data = region_data[:train_size]
    train_labels = region_labels[:train_size]
    train_prob_idx = region_prob_idx[:train_size]
    train_age = region_age[:train_size] if region_age is not None else None
    
    # 验证集
    val_data = region_data[train_size:]
    val_labels = region_labels[train_size:]
    val_prob_idx = region_prob_idx[train_size:]
    val_age = region_age[train_size:] if region_age is not None else None
    
    print(f"区域 {region_idx}: 总样本 {len(region_data)}，训练集 {len(train_data)}，验证集 {len(val_data)}")
    
    # 应用StandardScaler标准化
    train_data_scaled = scaler.transform(train_data)
    val_data_scaled = scaler.transform(val_data)
    
    # 1. 保存为mat格式
    # 训练集
    train_mat_file = os.path.join(train_dir, 'mat', f"region_{region_idx}.mat")
    train_dict = {
        'data': train_data_scaled,
        'region': train_labels,
        'prob_idx': train_prob_idx
    }
    if train_age is not None:
        train_dict['age'] = train_age
    scipy.io.savemat(train_mat_file, train_dict)
    
    # 验证集
    val_mat_file = os.path.join(val_dir, 'mat', f"region_{region_idx}.mat")
    val_dict = {
        'data': val_data_scaled,
        'region': val_labels,
        'prob_idx': val_prob_idx
    }
    if val_age is not None:
        val_dict['age'] = val_age
    scipy.io.savemat(val_mat_file, val_dict)
    
    # 2. 保存为npy格式
    # 训练集
    np.save(os.path.join(train_dir, 'npy', f"region_{region_idx}_data.npy"), train_data_scaled)
    np.save(os.path.join(train_dir, 'npy', f"region_{region_idx}_region.npy"), train_labels)
    np.save(os.path.join(train_dir, 'npy', f"region_{region_idx}_prob_idx.npy"), train_prob_idx)
    if train_age is not None:
        np.save(os.path.join(train_dir, 'npy', f"region_{region_idx}_age.npy"), train_age)
        print(f"已保存训练集区域 {region_idx} 的age数据，形状: {train_age.shape}")
    
    # 验证集
    np.save(os.path.join(val_dir, 'npy', f"region_{region_idx}_data.npy"), val_data_scaled)
    np.save(os.path.join(val_dir, 'npy', f"region_{region_idx}_region.npy"), val_labels)
    np.save(os.path.join(val_dir, 'npy', f"region_{region_idx}_prob_idx.npy"), val_prob_idx)
    if val_age is not None:
        np.save(os.path.join(val_dir, 'npy', f"region_{region_idx}_age.npy"), val_age)
        print(f"已保存验证集区域 {region_idx} 的age数据，形状: {val_age.shape}")

print("训练和验证集处理完成！")

In [ ]:
# 单元格6（修正版）：处理测试集数据，确保正确保存age数据
# 功能：按区域标签处理测试集数据，同时保存npy和mat格式

# 找出测试集中哪些区域标签有数据
test_active_regions = []
for i in range(test_regions.shape[1]):
    if np.sum(test_regions[:, i]) > 0:
        test_active_regions.append(i)

print(f"测试集活跃的区域标签数: {len(test_active_regions)}")
print(f"测试集活跃的区域标签: {test_active_regions}")

# 对每个活跃区域进行处理
for region_idx in tqdm(test_active_regions, desc="处理测试集区域标签"):
    # 找出该区域的所有体素
    indices = np.where(test_regions[:, region_idx] == 1)[0]
    
    if len(indices) == 0:
        print(f"测试集区域 {region_idx} 没有数据，跳过")
        continue
    
    # 提取该区域的所有数据
    region_data = test_data[indices]
    region_labels = test_regions[indices]
    region_prob_idx = test_prob_idx[indices]
    region_age = test_age[indices] if test_age is not None else None
    
    # 应用StandardScaler标准化
    region_data_scaled = scaler.transform(region_data)
    
    # 1. 保存为mat格式
    test_mat_file = os.path.join(test_dir, 'mat', f"region_{region_idx}.mat")
    test_dict = {
        'data': region_data_scaled,
        'region': region_labels,
        'prob_idx': region_prob_idx
    }
    if region_age is not None:
        test_dict['age'] = region_age
    scipy.io.savemat(test_mat_file, test_dict)
    
    # 2. 保存为npy格式
    np.save(os.path.join(test_dir, 'npy', f"region_{region_idx}_data.npy"), region_data_scaled)
    np.save(os.path.join(test_dir, 'npy', f"region_{region_idx}_region.npy"), region_labels)
    np.save(os.path.join(test_dir, 'npy', f"region_{region_idx}_prob_idx.npy"), region_prob_idx)
    if region_age is not None:
        np.save(os.path.join(test_dir, 'npy', f"region_{region_idx}_age.npy"), region_age)
        print(f"已保存测试集区域 {region_idx} 的age数据，形状: {region_age.shape}")
    
    print(f"测试集区域 {region_idx}: {len(region_data)} 个样本")
    
    # 可选：按病人ID分别保存
    unique_patients = np.unique(region_prob_idx)
    for patient_id in unique_patients:
        patient_indices = np.where(region_prob_idx == patient_id)[0]
        
        if len(patient_indices) == 0:
            continue
            
        patient_data = region_data_scaled[patient_indices]
        patient_labels = region_labels[patient_indices]
        patient_prob_idx = region_prob_idx[patient_indices]
        patient_age = region_age[patient_indices] if region_age is not None else None
        
        # 保存单个病人的数据，同时保存mat和npy格式
        # mat格式
        patient_mat_file = os.path.join(test_dir, 'mat', f"patient_{int(patient_id)}_region_{region_idx}.mat")
        patient_dict = {
            'data': patient_data,
            'region': patient_labels,
            'prob_idx': patient_prob_idx
        }
        if patient_age is not None:
            patient_dict['age'] = patient_age
        scipy.io.savemat(patient_mat_file, patient_dict)
        
        # npy格式
        patient_dir = os.path.join(test_dir, 'npy', f"patient_{int(patient_id)}")
        if not os.path.exists(patient_dir):
            os.makedirs(patient_dir)
        np.save(os.path.join(patient_dir, f"region_{region_idx}_data.npy"), patient_data)
        np.save(os.path.join(patient_dir, f"region_{region_idx}_region.npy"), patient_labels)
        np.save(os.path.join(patient_dir, f"region_{region_idx}_prob_idx.npy"), patient_prob_idx)
        if patient_age is not None:
            np.save(os.path.join(patient_dir, f"region_{region_idx}_age.npy"), patient_age)
            print(f"已保存病人 {int(patient_id)} 区域 {region_idx} 的age数据，形状: {patient_age.shape}")

print("测试集处理完成！")

In [ ]:
# 单元格7：创建索引文件
# 功能：为mat和npy格式分别创建索引文件

def create_mat_index(directory):
    """为mat格式文件创建索引"""
    mat_dir = os.path.join(directory, 'mat')
    index_file = os.path.join(mat_dir, "region_index.txt")
    
    with open(index_file, 'w') as f:
        f.write("region_id,sample_count,file_name,has_age\n")
        
        # 获取所有区域文件
        region_files = [file for file in os.listdir(mat_dir) 
                      if file.endswith('.mat') and file.startswith('region_')]
        
        for file in sorted(region_files, key=lambda x: int(x.split('_')[1].split('.')[0])):
            # 从文件名提取区域ID
            region_id = int(file.split('_')[1].split('.')[0])
            
            # 读取文件获取样本数和是否有年龄数据
            mat_data = scipy.io.loadmat(os.path.join(mat_dir, file))
            sample_count = mat_data['data'].shape[0]
            has_age = 'age' in mat_data
            
            f.write(f"{region_id},{sample_count},{file},{has_age}\n")
    
    print(f"Mat索引文件已创建: {index_file}")
    
    # 如果有按病人分类的文件，也为它们创建索引
    patient_files = [file for file in os.listdir(mat_dir) 
                    if file.endswith('.mat') and file.startswith('patient_')]
    
    if patient_files:
        patient_index_file = os.path.join(mat_dir, "patient_index.txt")
        
        with open(patient_index_file, 'w') as f:
            f.write("patient_id,region_id,sample_count,file_name,has_age\n")
            
            for file in sorted(patient_files, key=lambda x: (int(x.split('_')[1]), int(x.split('_')[3].split('.')[0]))):
                # 从文件名提取信息
                parts = file.split('_')
                patient_id = int(parts[1])
                region_id = int(parts[3].split('.')[0])
                
                # 读取文件获取样本数和是否有年龄数据
                mat_data = scipy.io.loadmat(os.path.join(mat_dir, file))
                sample_count = mat_data['data'].shape[0]
                has_age = 'age' in mat_data
                
                f.write(f"{patient_id},{region_id},{sample_count},{file},{has_age}\n")
        
        print(f"Mat病人索引文件已创建: {patient_index_file}")

def create_npy_index(directory):
    """为npy格式文件创建索引"""
    npy_dir = os.path.join(directory, 'npy')
    index_file = os.path.join(npy_dir, "region_index.txt")
    
    with open(index_file, 'w') as f:
        f.write("region_id,sample_count,data_file,region_file,prob_idx_file,age_file\n")
        
        # 获取所有区域数据文件
        data_files = [file for file in os.listdir(npy_dir) 
                     if file.endswith('_data.npy') and file.startswith('region_')]
        
        for file in sorted(data_files, key=lambda x: int(x.split('_')[1])):
            # 从文件名提取区域ID
            region_id = int(file.split('_')[1])
            
            # 构建对应的文件名
            base_name = f"region_{region_id}"
            region_file = f"{base_name}_region.npy"
            prob_idx_file = f"{base_name}_prob_idx.npy"
            age_file = f"{base_name}_age.npy"
            
            # 检查文件是否存在
            region_exists = os.path.exists(os.path.join(npy_dir, region_file))
            prob_idx_exists = os.path.exists(os.path.join(npy_dir, prob_idx_file))
            age_exists = os.path.exists(os.path.join(npy_dir, age_file))
            
            # 读取样本数
            data = np.load(os.path.join(npy_dir, file))
            sample_count = data.shape[0]
            
            f.write(f"{region_id},{sample_count},{file},{region_file if region_exists else 'N/A'},{prob_idx_file if prob_idx_exists else 'N/A'},{age_file if age_exists else 'N/A'}\n")
    
    print(f"Npy索引文件已创建: {index_file}")
    
    # 检查是否有按病人分类的目录
    patient_dirs = [d for d in os.listdir(npy_dir) if os.path.isdir(os.path.join(npy_dir, d)) and d.startswith('patient_')]
    
    if patient_dirs:
        patient_index_file = os.path.join(npy_dir, "patient_index.txt")
        
        with open(patient_index_file, 'w') as f:
            f.write("patient_id,region_id,sample_count,data_file,region_file,prob_idx_file,age_file\n")
            
            for patient_dir in sorted(patient_dirs, key=lambda x: int(x.split('_')[1])):
                patient_id = int(patient_dir.split('_')[1])
                patient_path = os.path.join(npy_dir, patient_dir)
                
                # 获取该病人目录下的所有数据文件
                data_files = [file for file in os.listdir(patient_path) if file.endswith('_data.npy')]
                
                for file in sorted(data_files, key=lambda x: int(x.split('_')[1])):
                    region_id = int(file.split('_')[1])
                    
                    # 构建对应的文件名
                    base_name = f"region_{region_id}"
                    region_file = f"{base_name}_region.npy"
                    prob_idx_file = f"{base_name}_prob_idx.npy"
                    age_file = f"{base_name}_age.npy"
                    
                    # 检查文件是否存在
                    region_exists = os.path.exists(os.path.join(patient_path, region_file))
                    prob_idx_exists = os.path.exists(os.path.join(patient_path, prob_idx_file))
                    age_exists = os.path.exists(os.path.join(patient_path, age_file))
                    
                    # 读取样本数
                    data = np.load(os.path.join(patient_path, file))
                    sample_count = data.shape[0]
                    
                    # 文件路径使用相对路径，便于移动
                    data_path = os.path.join(patient_dir, file)
                    region_path = os.path.join(patient_dir, region_file) if region_exists else 'N/A'
                    prob_idx_path = os.path.join(patient_dir, prob_idx_file) if prob_idx_exists else 'N/A'
                    age_path = os.path.join(patient_dir, age_file) if age_exists else 'N/A'
                    
                    f.write(f"{patient_id},{region_id},{sample_count},{data_path},{region_path},{prob_idx_path},{age_path}\n")
        
        print(f"Npy病人索引文件已创建: {patient_index_file}")

# 为每个数据集创建索引
print("创建数据集索引文件...")
for directory in [train_dir, val_dir, test_dir]:
    create_mat_index(directory)
    create_npy_index(directory)

In [ ]:
# 单元格8：创建数据加载辅助函数
# 功能：提供加载处理后数据的辅助函数，支持mat和npy格式

data_loader_file = os.path.join(output_base_dir, "data_loader.py")

with open(data_loader_file, 'w') as f:
    f.write("""# 数据加载辅助函数
import numpy as np
import os
import scipy.io
import h5py
import glob
import pickle

def load_scaler(base_dir):
    \"\"\"加载保存的StandardScaler\"\"\"
    scaler_path = os.path.join(base_dir, 'data_scaler.pkl')
    with open(scaler_path, 'rb') as f:
        return pickle.load(f)

def load_mat_region_index(directory):
    \"\"\"加载mat格式的区域索引文件\"\"\"
    index_file = os.path.join(directory, 'mat', "region_index.txt")
    index_data = {}
    
    if not os.path.exists(index_file):
        print(f"索引文件不存在: {index_file}")
        return index_data
    
    with open(index_file, 'r') as f:
        # 跳过标题行
        next(f)
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                region_id = int(parts[0])
                sample_count = int(parts[1])
                file_name = parts[2]
                has_age = parts[3].lower() == 'true' if len(parts) > 3 else False
                
                index_data[region_id] = {
                    'sample_count': sample_count,
                    'file_name': file_name,
                    'has_age': has_age
                }
    
    return index_data

def load_npy_region_index(directory):
    \"\"\"加载npy格式的区域索引文件\"\"\"
    index_file = os.path.join(directory, 'npy', "region_index.txt")
    index_data = {}
    
    if not os.path.exists(index_file):
        print(f"索引文件不存在: {index_file}")
        return index_data
    
    with open(index_file, 'r') as f:
        # 跳过标题行
        next(f)
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                region_id = int(parts[0])
                sample_count = int(parts[1])
                data_file = parts[2]
                region_file = parts[3] if parts[3] != 'N/A' else None
                prob_idx_file = parts[4] if parts[4] != 'N/A' else None
                age_file = parts[5] if len(parts) > 5 and parts[5] != 'N/A' else None
                
                index_data[region_id] = {
                    'sample_count': sample_count,
                    'data_file': data_file,
                    'region_file': region_file,
                    'prob_idx_file': prob_idx_file,
                    'age_file': age_file
                }
    
    return index_data

def load_mat_region_data(directory, region_id):
    \"\"\"
    加载mat格式的区域数据
    
    参数:
        directory: 数据目录路径
        region_id: 区域ID
    
    返回:
        包含data, region, prob_idx和可能的age的字典
    \"\"\"
    index_data = load_mat_region_index(directory)
    
    if region_id not in index_data:
        print(f"区域 {region_id} 在目录 {directory} 中不存在")
        return None
    
    file_name = index_data[region_id]['file_name']
    file_path = os.path.join(directory, 'mat', file_name)
    
    if not os.path.exists(file_path):
        print(f"文件不存在: {file_path}")
        return None
    
    try:
        # 尝试使用scipy.io.loadmat加载
        mat_data = scipy.io.loadmat(file_path)
    except:
        # 如果失败，尝试使用h5py加载
        with h5py.File(file_path, 'r') as f:
            mat_data = {}
            for key in f.keys():
                mat_data[key] = np.array(f[key])
    
    return mat_data

def load_npy_region_data(directory, region_id):
    \"\"\"
    加载npy格式的区域数据
    
    参数:
        directory: 数据目录路径
        region_id: 区域ID
    
    返回:
        包含data, region, prob_idx和可能的age的字典
    \"\"\"
    index_data = load_npy_region_index(directory)
    
    if region_id not in index_data:
        print(f"区域 {region_id} 在目录 {directory} 中不存在")
        return None
    
    info = index_data[region_id]
    npy_dir = os.path.join(directory, 'npy')
    
    result = {}
    
    # 加载数据
    data_path = os.path.join(npy_dir, info['data_file'])
    if os.path.exists(data_path):
        result['data'] = np.load(data_path)
    
    # 加载区域标签
    if info['region_file']:
        region_path = os.path.join(npy_dir, info['region_file'])
        if os.path.exists(region_path):
            result['region'] = np.load(region_path)
    
    # 加载病人ID
    if info['prob_idx_file']:
        prob_idx_path = os.path.join(npy_dir, info['prob_idx_file'])
        if os.path.exists(prob_idx_path):
            result['prob_idx'] = np.load(prob_idx_path)
    
    # 加载年龄数据（如果有）
    if info['age_file']:
        age_path = os.path.join(npy_dir, info['age_file'])
        if os.path.exists(age_path):
            result['age'] = np.load(age_path)
    
    return result

def load_region_data(directory, region_id, format='mat'):
    \"\"\"
    加载区域数据，支持mat和npy格式
    
    参数:
        directory: 数据目录路径
        region_id: 区域ID
        format: 'mat'或'npy'，指定加载格式
    
    返回:
        包含data, region, prob_idx和可能的age的字典
    \"\"\"
    if format.lower() == 'mat':
        return load_mat_region_data(directory, region_id)
    elif format.lower() == 'npy':
        return load_npy_region_data(directory, region_id)
    else:
        print(f"不支持的格式: {format}")
        return None

def load_all_regions(directory, region_ids=None, format='mat'):
    \"\"\"
    加载指定目录下的所有区域数据或指定区域数据
    
    参数:
        directory: 数据目录路径
        region_ids: 要加载的区域ID列表，如果为None则加载所有区域
        format: 'mat'或'npy'，指定加载格式
    
    返回:
        包含所有区域数据的字典，键为区域ID
    \"\"\"
    if format.lower() == 'mat':
        index_data = load_mat_region_index(directory)
    elif format.lower() == 'npy':
        index_data = load_npy_region_index(directory)
    else:
        print(f"不支持的格式: {format}")
        return {}
    
    if region_ids is None:
        region_ids = list(index_data.keys())
    
    result = {}
    for region_id in region_ids:
        if region_id in index_data:
            region_data = load_region_data(directory, region_id, format)
            if region_data is not None:
                result[region_id] = region_data
    
    return result

def load_patient_data(directory, patient_id, region_id=None, format='mat'):
    \"\"\"
    加载指定病人的数据
    
    参数:
        directory: 数据目录路径
        patient_id: 病人ID
        region_id: 区域ID（可选，如果只想加载特定区域的数据）
        format: 'mat'或'npy'，指定加载格式
    
    返回:
        包含该病人数据的字典，键为区域ID
    \"\"\"
    if format.lower() == 'mat':
        patient_index_file = os.path.join(directory, 'mat', "patient_index.txt")
        
        if not os.path.exists(patient_index_file):
            print(f"病人索引文件不存在: {patient_index_file}")
            return {}
        
        patient_data = {}
        
        with open(patient_index_file, 'r') as f:
            # 跳过标题行
            next(f)
            for line in f:
                parts = line.strip().split(',')
                if len(parts) >= 4:
                    curr_patient_id = int(parts[0])
                    curr_region_id = int(parts[1])
                    file_name = parts[3]
                    
                    if curr_patient_id == patient_id and (region_id is None or curr_region_id == region_id):
                        file_path = os.path.join(directory, 'mat', file_name)
                        
                        try:
                            # 尝试使用scipy.io.loadmat加载
                            mat_data = scipy.io.loadmat(file_path)
                        except:
                            # 如果失败，尝试使用h5py加载
                            with h5py.File(file_path, 'r') as f:
                                mat_data = {}
                                for key in f.keys():
                                    mat_data[key] = np.array(f[key])
                        
                        patient_data[curr_region_id] = mat_data
        
        return patient_data
    
    elif format.lower() == 'npy':
        npy_dir = os.path.join(directory, 'npy')
        patient_dir = os.path.join(npy_dir, f"patient_{patient_id}")
        
        if not os.path.exists(patient_dir):
            print(f"病人目录不存在: {patient_dir}")
            return {}
        
        patient_data = {}
        
        # 获取该病人目录下的所有数据文件
        data_files = [file for file in os.listdir(patient_dir) if file.endswith('_data.npy')]
        
        for file in data_files:
            curr_region_id = int(file.split('_')[1])
            
            if region_id is None or curr_region_id == region_id:
                # 构建数据字典
                region_data = {}
                
                # 加载数据
                data_path = os.path.join(patient_dir, file)
                region_data['data'] = np.load(data_path)
                
                # 尝试加载其他文件
                base_name = f"region_{curr_region_id}"
                region_file = f"{base_name}_region.npy"
                prob_idx_file = f"{base_name}_prob_idx.npy"
                age_file = f"{base_name}_age.npy"
                
                region_path = os.path.join(patient_dir, region_file)
                if os.path.exists(region_path):
                    region_data['region'] = np.load(region_path)
                
                prob_idx_path = os.path.join(patient_dir, prob_idx_file)
                if os.path.exists(prob_idx_path):
                    region_data['prob_idx'] = np.load(prob_idx_path)
                
                age_path = os.path.join(patient_dir, age_file)
                if os.path.exists(age_path):
                    region_data['age'] = np.load(age_path)
                
                patient_data[curr_region_id] = region_data
        
        return patient_data
    
    else:
        print(f"不支持的格式: {format}")
        return {}

def load_dataset(base_dir, split='train', region_ids=None, format='mat'):
    \"\"\"
    加载指定数据集
    
    参数:
        base_dir: 基础目录路径
        split: 数据集类型，'train', 'val'或'test'
        region_ids: 要加载的区域ID列表，如果为None则加载所有区域
        format: 'mat'或'npy'，指定加载格式
    
    返回:
        包含指定数据集所有区域数据的字典
    \"\"\"
    directory = os.path.join(base_dir, split)
    return load_all_regions(directory, region_ids, format)

def concatenate_regions(regions_data):
    \"\"\"
    将多个区域的数据合并成一个大数组
    
    参数:
        regions_data: 由load_all_regions返回的区域数据字典
    
    返回:
        包含合并后数据的字典，键为'data', 'region', 'prob_idx'和可能的'age'
    \"\"\"
    all_data = []
    all_region = []
    all_prob_idx = []
    all_age = []
    has_age = False
    
    for region_id, region_data in regions_data.items():
        if 'data' in region_data:
            all_data.append(region_data['data'])
        
        if 'region' in region_data:
            all_region.append(region_data['region'])
        
        if 'prob_idx' in region_data:
            all_prob_idx.append(region_data['prob_idx'])
        
        if 'age' in region_data:
            all_age.append(region_data['age'])
            has_age = True
    
    result = {}
    
    if all_data:
        result['data'] = np.vstack(all_data)
    
    if all_region:
        result['region'] = np.vstack(all_region)
    
    if all_prob_idx:
        result['prob_idx'] = np.vstack(all_prob_idx)
    
    if has_age and all_age:
        result['age'] = np.vstack(all_age)
    
    return result

# 使用示例
if __name__ == "__main__":
    # 替换为实际路径
    base_dir = "processed_data"
    
    # 加载Scaler
    scaler = load_scaler(base_dir)
    print(f"Scaler已加载，特征数量: {len(scaler.mean_)}")
    
    # 加载训练集中的某个区域，mat格式
    region_id = 0  # 替换为实际区域ID
    region_data_mat = load_region_data(os.path.join(base_dir, 'train'), region_id, format='mat')
    if region_data_mat:
        print(f"区域 {region_id} 的训练数据 (mat格式):")
        for key, value in region_data_mat.items():
            print(f"  - {key}: 形状 {value.shape}")
    
    # 加载训练集中的某个区域，npy格式
    region_data_npy = load_region_data(os.path.join(base_dir, 'train'), region_id, format='npy')
    if region_data_npy:
        print(f"区域 {region_id} 的训练数据 (npy格式):")
        for key, value in region_data_npy.items():
            print(f"  - {key}: 形状 {value.shape}")
    
    # 加载测试集中特定病人的数据
    patient_id = 38  # 替换为实际病人ID
    patient_data_mat = load_patient_data(os.path.join(base_dir, 'test'), patient_id, format='mat')
    print(f"病人 {patient_id} 在测试集中的数据 (mat格式):")
    for region_id, data in patient_data_mat.items():
        print(f"  - 区域 {region_id}: {data['data'].shape[0]} 个样本")
""")

print(f"数据加载辅助函数已保存到: {data_loader_file}")

In [ ]:
# 单元格9：验证处理结果
# 功能：验证处理后的数据集是否正确保存和索引

print("验证处理结果...")

# 验证输出目录结构
print("1. 验证目录结构:")
for directory in [train_dir, val_dir, test_dir]:
    mat_files = [f for f in os.listdir(directory) if f.endswith('.mat')]
    region_files = [f for f in mat_files if f.startswith('region_')]
    patient_files = [f for f in mat_files if f.startswith('patient_')]
    
    print(f"  - {os.path.basename(directory)}目录:")
    print(f"    * {len(region_files)} 个区域数据文件")
    print(f"    * {len(patient_files)} 个病人数据文件")

# 验证索引文件
print("\n2. 验证索引文件:")
for directory in [train_dir, val_dir, test_dir]:
    index_file = os.path.join(directory, "region_index.txt")
    if os.path.exists(index_file):
        with open(index_file, 'r') as f:
            headers = next(f).strip().split(',')
            line_count = sum(1 for _ in f)
        print(f"  - {os.path.basename(directory)}区域索引文件: {line_count} 个条目")
        print(f"    * 索引字段: {headers}")
    
    patient_index_file = os.path.join(directory, "patient_index.txt")
    if os.path.exists(patient_index_file):
        with open(patient_index_file, 'r') as f:
            headers = next(f).strip().split(',')
            line_count = sum(1 for _ in f)
        print(f"  - {os.path.basename(directory)}病人索引文件: {line_count} 个条目")
        print(f"    * 索引字段: {headers}")

# 验证数据文件内容
print("\n3. 验证数据文件内容:")
for directory in [train_dir, val_dir, test_dir]:
    region_files = [f for f in os.listdir(directory) if f.endswith('.mat') and f.startswith('region_')]
    
    if region_files:
        sample_file = os.path.join(directory, region_files[0])
        with h5py.File(sample_file, 'r') as f:
            keys = list(f.keys())
            data_shape = f['data'].shape if 'data' in f else None
            region_shape = f['region'].shape if 'region' in f else None
            prob_idx_shape = f['prob_idx'].shape if 'prob_idx' in f else None
            age_shape = f['age'].shape if 'age' in f else None
            
            print(f"  - {os.path.basename(directory)}样本文件 {os.path.basename(sample_file)}:")
            print(f"    * 包含键: {keys}")
            print(f"    * 数据形状: {data_shape}")
            print(f"    * 标签形状: {region_shape}")
            print(f"    * 病人ID形状: {prob_idx_shape}")
            if age_shape:
                print(f"    * 年龄数据形状: {age_shape}")

# 验证Scaler
print("\n4. 验证Scaler:")
try:
    with open(scaler_path, 'rb') as f:
        loaded_scaler = pickle.load(f)
    
    print(f"  - Scaler加载成功, 特征数量: {len(loaded_scaler.mean_)}")
except Exception as e:
    print(f"  - Scaler验证出错: {str(e)}")

print("\n处理完成！所有数据已按要求按区域标签分别处理并保存，保持了数据结构的完整性。")

In [ ]:
# 单元格11：使用说明和总结
print("""
================================================================================
                           数据处理完成总结
================================================================================

处理流程概述:
1. 从TRAIN38.mat文件中加载数据，保留完整的原始结构
2. 将病人分配到测试集和训练+验证集
3. 使用训练和验证集数据拟合StandardScaler并保存
4. 按照102个区域标签分别处理数据，保持数据结构完整性
5. 训练和验证集数据按6:2比例拆分
6. 所有数据同时保存为.mat和.npy格式，便于不同场景使用
7. 创建索引文件和辅助函数便于使用

输出文件:
- 训练集数据: {}/train/mat/region_*.mat 和 {}/train/npy/region_*_*.npy
- 验证集数据: {}/val/mat/region_*.mat 和 {}/val/npy/region_*_*.npy
- 测试集数据: {}/test/mat/region_*.mat 和 {}/test/npy/region_*_*.npy
- 测试集按病人分类数据: {}/test/mat/patient_*_region_*.mat 和 {}/test/npy/patient_*/region_*_*.npy
- 数据标准化器: {}/data_scaler.pkl
- 处理汇总信息: {}/processing_summary.txt
- 数据加载辅助函数: {}/data_loader.py

文件格式说明:
1. Mat格式 (.mat):
   - 每个文件包含完整的data, region, prob_idx和可能的age
   - 适合需要完整数据结构的场景
   - 可以使用scipy.io.loadmat或h5py加载

2. Npy格式 (.npy):
   - 每个区域的data, region, prob_idx和age分别保存为单独的文件
   - 适合需要高效加载特定数据的场景
   - 使用np.load直接加载

使用数据的方法:
1. 使用提供的data_loader.py加载数据，支持mat和npy两种格式:
   - load_region_data(directory, region_id, format='mat'): 加载指定区域的数据
   - load_all_regions(directory, region_ids=None, format='mat'): 加载所有区域的数据
   - load_patient_data(directory, patient_id, region_id=None, format='mat'): 加载指定病人的数据
   - load_dataset(base

In [ ]:
# 单元格11：总结和使用说明
print("""
================================================================================
                           数据处理完成总结
================================================================================

处理流程概述:
1. 从TRAIN38.mat文件中加载数据，保留完整的原始结构
2. 将病人分配到测试集和训练+验证集
3. 使用训练和验证集数据拟合StandardScaler并保存
4. 按照102个区域标签分别处理数据，保持数据结构完整性
5. 训练和验证集数据按6:2比例拆分
6. 所有数据保存为.mat文件，保持原始数据的键结构
7. 创建索引文件和辅助函数便于使用

输出文件:
- 训练集数据: {}/train/region_*.mat
- 验证集数据: {}/val/region_*.mat
- 测试集数据: {}/test/region_*.mat
- 测试集按病人分类数据: {}/test/patient_*_region_*.mat
- 数据标准化器: {}/data_scaler.pkl
- 处理汇总信息: {}/processing_summary.txt
- 数据加载辅助函数: {}/data_loader.py

数据文件结构:
每个.mat文件都保留了与原始TRAIN38.mat相同的键结构:
- data: 标准化后的体素特征数据
- region: 102维的区域标签
- prob_idx: 病人ID
- age: 年龄数据（如果原始数据中存在）

使用数据的方法:
1. 使用提供的data_loader.py加载数据:
   - load_region_data(): 加载指定区域的数据
   - load_all_regions(): 加载所有区域的数据
   - load_patient_data(): 加载指定病人的数据
   - load_dataset(base_dir, split='train', region_ids=None, format='mat'): 加载指定数据集
   - concatenate_regions(regions_data): 将多个区域的数据合并

2. 使用保存的Scaler对新数据进行标准化:
   ```python
   from data_loader import load_scaler
   scaler = load_scaler(base_dir)
   new_data_scaled = scaler.transform(new_data)



"""
)


# 单元格12：检查age数据结构
# 功能：检查TRAIN38.mat中age数据的结构并分析问题

import h5py
import numpy as np

# 加载原始TRAIN38.mat文件
print("加载原始数据文件以检查age字段...")
f = h5py.File(data_path, 'r')

# 检查文件中包含的所有键
keys = list(f.keys())
print(f"TRAIN38.mat包含的键: {keys}")

# 特别检查age键
if 'age' in f:
    age_data = f['age']
    age_shape = age_data.shape
    print(f"age数据形状: {age_shape}")
    
    # 提取age数据样本
    age_sample = np.array(age_data)[:10]  # 取前10个样本
    print(f"age数据样本: {age_sample}")
    
    # 检查age是否是标量还是向量
    if len(age_shape) > 1:
        print(f"age是多维数组，维度为: {len(age_shape)}")
        # 如果是二维数组，检查第二个维度的大小
        if len(age_shape) == 2:
            print(f"第二维度大小: {age_shape[1]}")
    else:
        print("age是一维数组")
    
    # 检查age数据的数据类型
    age_dtype = age_data.dtype
    print(f"age数据类型: {age_dtype}")
    
    # 检查age数据是否与prob_idx长度一致
    if 'prob_idx' in f:
        prob_idx_shape = f['prob_idx'].shape
        print(f"prob_idx形状: {prob_idx_shape}")
        if age_shape[0] == prob_idx_shape[0]:
            print("age和prob_idx长度一致")
        else:
            print("警告: age和prob_idx长度不一致!")
else:
    print("警告: 原始数据文件中不包含'age'键!")

# 关闭文件
f.close()

# 如果有必要，检查处理后的age数据
if 'age' in locals() and age is not None:
    print("\n检查处理后的age变量...")
    print(f"处理后的age形状: {age.shape}")
    print(f"处理后的age样本: {age[:10]}")
    print(f"处理后的age数据类型: {age.dtype}")
else:
    print("\n处理过程中未正确加载age数据或age数据不存在")

# 提供一个修复函数，用于正确保存age数据
def fix_age_data_saving():
    """修复age数据保存问题的函数"""
    print("\n开始修复age数据保存...")
    
    # 重新加载原始数据以获取age
    with h5py.File(data_path, 'r') as f:
        if 'age' in f:
            original_age = np.array(f['age'])
            if len(original_age.shape) > 1:
                # 如果age是多维的，需要转置
                original_age = original_age.transpose()
            print(f"重新加载的age数据形状: {original_age.shape}")
            
            # 根据之前的分割重新分配age数据
            test_age = original_age[test_indices] if len(test_indices) > 0 else None
            train_val_age = original_age[train_val_indices] if len(train_val_indices) > 0 else None
            
            print(f"测试集age形状: {test_age.shape if test_age is not None else 'None'}")
            print(f"训练验证集age形状: {train_val_age.shape if train_val_age is not None else 'None'}")
            
            return original_age, test_age, train_val_age
        else:
            print("原始数据文件中不包含'age'键，无法修复")
            return None, None, None

# 执行修复函数
original_age, fixed_test_age, fixed_train_val_age = fix_age_data_saving()

# 显示修复建议
if original_age is not None:
    print("\n修复建议:")
    print("1. 确认原始数据中的age字段结构和转置方式")
    print("2. 修改单元格2中加载age的代码，确保正确转置")
    print("3. 修改单元格5和6中的保存代码，确保age数据正确保存")
    print("例如，修改加载代码为:")
    print("   age = arrays['age'].transpose() if 'age' in arrays else None")
    print("然后在保存npy文件时检查age是否为None")

In [ ]:
#pip install torch torchvision tqdm matplotlib scikit-learn numpy

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import pickle
from tqdm.auto import tqdm
import pandas as pd
import seaborn as sns

# 导入您的data_loader
from data_loader import load_brain_voxel_data, load_scaler

# 设置随机种子以确保可重现性
torch.manual_seed(42)
np.random.seed(42)

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

# 模型超参数 (与原始模型保持一致)
batch_size = 128
no_epochs = 25
no_classes = 102
learning_rate = 0.00001
weight_decay = 0.00001  # 对应L2正则化参数
input_dim = 341
dropout_rate = 0.5

# 定义与原始模型结构相同的PyTorch模型
class DenseModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=4096, num_classes=102, dropout_rate=0.5):
        super(DenseModel, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer4 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.output_layer = nn.Linear(hidden_dim, num_classes)
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        logits = self.output_layer(x)
        return self.softmax(logits)

# 加载和准备数据
def prepare_data(base_dir):
    print("加载训练集...")
    train_data = load_brain_voxel_data(
        base_dir=base_dir, 
        split='train', 
        format='mat',
        shuffle=True,  # 全局打乱数据
        seed=666
    )
    
    print("加载验证集...")
    val_data = load_brain_voxel_data(
        base_dir=base_dir, 
        split='val', 
        format='mat',
        shuffle=False,  # 验证集不需要打乱
        seed=666
    )
    
    print("加载测试集...")
    test_data = load_brain_voxel_data(
        base_dir=base_dir, 
        split='test', 
        format='mat',
        shuffle=False,  # 测试集不需要打乱
        seed=666
    )
    
    # 加载保存的StandardScaler
    scaler = load_scaler(base_dir)
    print(f"加载StandardScaler成功, 特征数量: {len(scaler.mean_)}")
    
    # 转换为PyTorch张量
    X_train = torch.FloatTensor(train_data['features'])
    y_train = torch.tensor(train_data['labels'].astype(np.int64))
    
    X_val = torch.FloatTensor(val_data['features'])
    y_val = torch.tensor(val_data['labels'].astype(np.int64))
    
    X_test = torch.FloatTensor(test_data['features'])
    y_test = torch.tensor(test_data['labels'].astype(np.int64))
    
    # 转换为one-hot编码
    y_train_onehot = torch.zeros(y_train.size(0), no_classes)
    y_train_onehot.scatter_(1, y_train.unsqueeze(1), 1)
    
    y_val_onehot = torch.zeros(y_val.size(0), no_classes)
    y_val_onehot.scatter_(1, y_val.unsqueeze(1), 1)
    
    y_test_onehot = torch.zeros(y_test.size(0), no_classes)
    y_test_onehot.scatter_(1, y_test.unsqueeze(1), 1)
    
    # 创建数据集和数据加载器
    train_dataset = TensorDataset(X_train, y_train_onehot)
    val_dataset = TensorDataset(X_val, y_val_onehot)
    test_dataset = TensorDataset(X_test, y_test_onehot)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"训练集样本数: {len(X_train)}")
    print(f"验证集样本数: {len(X_val)}")
    print(f"测试集样本数: {len(X_test)}")
    
    # 检查是否包含第38号患者
    if 'prob_idx' in test_data:
        patient_38_count = np.sum(test_data['prob_idx'] == 38)
        print(f"测试集中第38号患者的样本数: {patient_38_count}")
    
    return train_loader, val_loader, test_loader, scaler

# 训练函数
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs, device):
    model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'train_f1': [], 'val_f1': []}
    
    for epoch in range(epochs):
        # 训练阶段
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        all_train_preds = []
        all_train_targets = []
        
        for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]"):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # 前向传播
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # 反向传播和优化
            loss.backward()
            optimizer.step()
            
            # 统计
            train_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            _, target_indices = torch.max(targets.data, 1)
            train_total += targets.size(0)
            train_correct += (predicted == target_indices).sum().item()
            
            # 收集预测和目标用于计算F1
            all_train_preds.extend(predicted.cpu().numpy())
            all_train_targets.extend(target_indices.cpu().numpy())
        
        train_loss = train_loss / train_total
        train_acc = train_correct / train_total
        train_f1 = f1_score(all_train_targets, all_train_preds, average='macro')
        
        # 验证阶段
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_val_preds = []
        all_val_targets = []
        
        with torch.no_grad():
            for inputs, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Validation]"):
                inputs, targets = inputs.to(device), targets.to(device)
                
                # 前向传播
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                
                # 统计
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs.data, 1)
                _, target_indices = torch.max(targets.data, 1)
                val_total += targets.size(0)
                val_correct += (predicted == target_indices).sum().item()
                
                # 收集预测和目标用于计算F1
                all_val_preds.extend(predicted.cpu().numpy())
                all_val_targets.extend(target_indices.cpu().numpy())
        
        val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        val_f1 = f1_score(all_val_targets, all_val_preds, average='macro')
        
        # 记录历史
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['train_f1'].append(train_f1)
        history['val_f1'].append(val_f1)
        
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Train F1: {train_f1:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, Val F1: {val_f1:.4f}")
    
    return history

# 测试函数
def test_model(model, test_loader, device):
    model.eval()
    test_correct = 0
    test_total = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc="Testing"):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # 前向传播
            outputs = model(inputs)
            
            # 统计
            _, predicted = torch.max(outputs.data, 1)
            _, target_indices = torch.max(targets.data, 1)
            test_total += targets.size(0)
            test_correct += (predicted == target_indices).sum().item()
            
            # 收集预测和目标用于计算F1
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(target_indices.cpu().numpy())
    
    test_acc = test_correct / test_total
    test_f1 = f1_score(all_targets, all_preds, average='macro')
    
    print(f"测试集准确率: {test_acc:.4f}, 测试集Macro F1: {test_f1:.4f}")
    
    # 计算每个类别的F1
    class_f1 = f1_score(all_targets, all_preds, average=None)
    
    # 打印分类报告
    report = classification_report(all_targets, all_preds)
    print("\n分类报告:")
    print(report)
    
    # 计算混淆矩阵
    cm = confusion_matrix(all_targets, all_preds)
    
    return test_acc, test_f1, class_f1, cm, all_preds, all_targets, report

# 可视化训练历史
def plot_history(history):
    plt.figure(figsize=(15, 10))
    
    # 绘制损失曲线
    plt.subplot(2, 2, 1)
    plt.plot(history['train_loss'], label='Training Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Loss Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # 绘制准确率曲线
    plt.subplot(2, 2, 2)
    plt.plot(history['train_acc'], label='Training Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Accuracy Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    # 绘制F1曲线
    plt.subplot(2, 2, 3)
    plt.plot(history['train_f1'], label='Training F1')
    plt.plot(history['val_f1'], label='Validation F1')
    plt.title('Macro F1 Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.show()

# 可视化混淆矩阵
def plot_confusion_matrix(cm, num_classes=102):
    plt.figure(figsize=(15, 15))
    
    # 只选择有预测的类别
    active_indices = np.unique(np.nonzero(cm)[0])
    active_cm = cm[np.ix_(active_indices, active_indices)]
    
    # 绘制热力图
    sns.heatmap(active_cm, annot=False, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix for Active Classes')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.savefig('confusion_matrix.png')
    plt.show()
    
    # 计算和保存每个类别的性能
    class_performance = []
    for i in range(num_classes):
        if i in active_indices:
            true_positive = cm[i, i]
            false_positive = cm[:, i].sum() - true_positive
            false_negative = cm[i, :].sum() - true_positive
            
            precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
            recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            
            class_performance.append({
                'class': i,
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'support': cm[i, :].sum()
            })
    
    # 将性能数据保存为CSV
    df = pd.DataFrame(class_performance)
    df.to_csv('class_performance.csv', index=False)
    
    return df


In [ ]:

# 设置数据基础目录
base_dir = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/processed_data"  # 请替换为您的目录

# 准备数据
train_loader, val_loader, test_loader, scaler = prepare_data(base_dir)

# 创建模型
model = DenseModel(input_dim=input_dim, hidden_dim=4096, num_classes=no_classes, dropout_rate=dropout_rate)
print(model)

# 定义损失函数和优化器
criterion = nn.BCELoss()  # 二元交叉熵损失，适用于one-hot编码
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)  # weight_decay对应L2正则化

# 训练模型
history = train_model(model, train_loader, val_loader, optimizer, criterion, no_epochs, device)

# 可视化训练历史
plot_history(history)

# 保存模型
torch.save(model.state_dict(), 'dense_4x4096_model_pytorch.pth')
print("模型已保存到 'dense_4x4096_model_pytorch.pth'")

# 保存训练历史记录
with open('training_history.pkl', 'wb') as f:
    pickle.dump(history, f)
print("训练历史已保存到 'training_history.pkl'")

# 在测试集上评估模型
print("\n在测试集上评估模型...")
test_acc, test_f1, class_f1, cm, all_preds, all_targets, report = test_model(model, test_loader, device)

# 保存测试结果
test_results = {
    'accuracy': test_acc,
    'macro_f1': test_f1,
    'class_f1': class_f1,
    'confusion_matrix': cm,
    'predictions': all_preds,
    'targets': all_targets,
    'classification_report': report
}

with open('test_results.pkl', 'wb') as f:
    pickle.dump(test_results, f)
print("测试结果已保存到 'test_results.pkl'")

# 可视化混淆矩阵
class_performance_df = plot_confusion_matrix(cm)

# 打印每个类别的F1分数
print("\n每个类别的F1分数:")
for i, f1 in enumerate(class_f1):
    print(f"类别 {i}: {f1:.4f}")

# 打印前10个性能最好和最差的类别
print("\n性能最好的10个类别:")
print(class_performance_df.sort_values('f1', ascending=False).head(10))

print("\n性能最差的10个类别:")
print(class_performance_df.sort_values('f1', ascending=True).head(10))

# 用于比较不同模型的函数
def compare_with_keras_model(pytorch_results_path, keras_results_path=None):
"""比较PyTorch模型和Keras模型的性能"""
# 加载PyTorch模型结果
with open(pytorch_results_path, 'rb') as f:
    pytorch_results = pickle.load(f)

print("PyTorch模型性能:")
print(f"准确率: {pytorch_results['accuracy']:.4f}")
print(f"Macro F1: {pytorch_results['macro_f1']:.4f}")

# 如果有Keras模型结果，则进行比较
if keras_results_path and os.path.exists(keras_results_path):
    with open(keras_results_path, 'rb') as f:
        keras_results = pickle.load(f)
    
    print("\nKeras模型性能:")
    print(f"准确率: {keras_results['accuracy']:.4f}")
    print(f"Macro F1: {keras_results['macro_f1']:.4f}")
    
    print("\n性能差异 (PyTorch - Keras):")
    print(f"准确率差异: {pytorch_results['accuracy'] - keras_results['accuracy']:.4f}")
    print(f"Macro F1差异: {pytorch_results['macro_f1'] - keras_results['macro_f1']:.4f}")
    
    # 如果两个模型都有类别F1，绘制比较图
    if 'class_f1' in pytorch_results and 'class_f1' in keras_results:
        plt.figure(figsize=(15, 8))
        
        # 找出两个模型都有预测的类别
        common_classes = np.intersect1d(
            np.nonzero(pytorch_results['class_f1'])[0],
            np.nonzero(keras_results['class_f1'])[0]
        )
        
        # 绘制每个类别的F1比较
        x = np.arange(len(common_classes))
        width = 0.35
        
        plt.bar(x - width/2, pytorch_results['class_f1'][common_classes], width, label='PyTorch')
        plt.bar(x + width/2, keras_results['class_f1'][common_classes], width, label='Keras')
        
        plt.xlabel('Class')
        plt.ylabel('F1 Score')
        plt.title('F1 Score Comparison by Class')
        plt.xticks(x, common_classes)
        plt.legend()
        plt.savefig('model_comparison.png')
        plt.show()
